In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data = pd.read_csv('/content/IMDB Dataset.csv',sep=',',on_bad_lines='skip', engine='python', quotechar='"')

In [ ]:
data

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
35626,Unfortunately there was not a 0 for a rating o...,negative
35627,This is a failure so complete as to make me an...,negative
35628,I had to compare two versions of Hamlet for my...,negative
35629,I was still living with my parents when they a...,positive


In [ ]:
data.shape

(35631, 2)

In [ ]:
data.columns

Index(['review', 'sentiment'], dtype='object')

In [ ]:
data.loc[0,'review']

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [ ]:
data.loc[:,'sentiment'].unique()

array(['positive', 'negative'], dtype=object)

In [ ]:
# convert positive-->1 , negative-->0
data.loc[:,'Sentiment']=data.loc[:,'sentiment'].map({'positive':1,'negative':0})

In [ ]:
data.drop(columns=['sentiment'],inplace=True)

#Basic Data Cleaning

In [ ]:
import re
def clean_text(text):
  text = text.lower()
  text = re.sub(r'<.*?>','',text)
  text =re.sub(r'[^a-zA-Z!?]',' ',text)
  text = re.sub(r'\s+',' ',text).strip()
  return text

In [ ]:
data.loc[:,'cleaned_text']=data.loc[:,'review'].apply(clean_text)

In [ ]:
data.loc[20000,'cleaned_text']

'i am a huge fan of northern exposure men in trees is a complete knock off of northern exposure there s the city folk from new york stuck in a remote backwoods town marin frist joel fleishman she immediately doesn t hit it off with a local jack slattery maggie o connell the town only has one pilot who is not often available ther is only one radio show and the entire show is about quirky people with sincere and odd relationships too many parallels just like northern exposure except one obvious demographic has been altered in each character i m bored give me something new to think about on the upside the writing is pretty good but if this was a school project it would certainly have earned the stamp of plagiarism perhaps people who love this show were sheltered from the dedicated creativity and hard labors of joshua brand and john falsey'

#Tokenizer

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(data['cleaned_text'])
X = tokenizer.texts_to_sequences(data['cleaned_text'])

In [ ]:
X[0]

[28,
 4,
 1,
 77,
 2030,
 46,
 1046,
 11,
 99,
 148,
 41,
 3084,
 410,
 20,
 228,
 29,
 3196,
 32,
 24,
 198,
 14,
 10,
 6,
 603,
 47,
 598,
 16,
 68,
 1,
 87,
 149,
 11,
 3179,
 68,
 43,
 3084,
 13,
 91,
 2,
 135,
 4,
 560,
 61,
 267,
 8,
 198,
 37,
 1,
 644,
 141,
 1705,
 68,
 10,
 6,
 23,
 3,
 118,
 17,
 1,
 2170,
 40,
 10,
 118,
 2427,
 56,
 16,
 5,
 1422,
 374,
 40,
 560,
 91,
 6,
 3645,
 8,
 1,
 356,
 354,
 4,
 1,
 644,
 7,
 6,
 434,
 3084,
 14,
 11,
 6,
 1,
 353,
 5,
 1,
 2538,
 1039,
 7,
 2698,
 1410,
 22,
 511,
 34,
 4808,
 2490,
 4,
 1,
 1139,
 116,
 30,
 1,
 27,
 2834,
 2,
 386,
 36,
 6,
 23,
 305,
 22,
 1,
 4809,
 2703,
 511,
 6,
 348,
 5,
 107,
 4557,
 2410,
 2,
 51,
 36,
 323,
 2,
 24,
 110,
 230,
 238,
 9,
 60,
 133,
 1,
 283,
 1286,
 4,
 1,
 118,
 6,
 676,
 5,
 1,
 191,
 11,
 7,
 270,
 116,
 77,
 274,
 582,
 21,
 2992,
 810,
 182,
 1333,
 4161,
 17,
 2444,
 1185,
 810,
 1404,
 810,
 855,
 3084,
 155,
 21,
 949,
 187,
 1,
 87,
 410,
 9,
 123,
 211,
 3179,
 68,
 14,
 36,


In [ ]:
y = data['Sentiment']

In [ ]:
# split data for training and testing
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
# padding is used to make length of sentences same
from tensorflow.keras.preprocessing.sequence import pad_sequences
x_train = pad_sequences(x_train,maxlen=100,padding='post',truncating='post')
x_test = pad_sequences(x_test,maxlen=100,padding='post',truncating='post')

#RNN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=5000,output_dim=128))
model.build(input_shape=(None,100))
model.add(SimpleRNN(64))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 652,417 (2.49 MB)

 Trainable params: 652,417 (2.49 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(loss='binary_crossentropy',optimizer='Adam',metrics=['accuracy'])

In [ ]:
model.fit(x_train,y_train,epochs=10,batch_size=64,verbose=1)

Epoch 1/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 21s 42ms/step - accuracy: 0.5228 - loss: 0.6926
Epoch 2/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.6479 - loss: 0.6275
Epoch 3/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.7142 - loss: 0.5565
Epoch 4/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 22s 49ms/step - accuracy: 0.7877 - loss: 0.4348
Epoch 5/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 40s 46ms/step - accuracy: 0.8384 - loss: 0.3459
Epoch 6/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.8621 - loss: 0.3137
Epoch 7/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 21s 45ms/step - accuracy: 0.9004 - loss: 0.2183
Epoch 8/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 24s 54ms/step - accuracy: 0.9365 - loss: 0.1272
Epoch 9/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 18s 41ms/step - accuracy: 0.9523 - loss: 0.0901
Epoch 10/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.9607 - loss: 0.0784


In [ ]:
loss,accuracy = model.evaluate(x_test,y_test)
print("Loss: ",loss)
print("Accuracy: ",accuracy)

223/223 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.5509 - loss: 2.0357
Loss:  2.035689115524292
Accuracy:  0.5508629083633423


#LSTM

In [ ]:
from tensorflow.keras.layers import LSTM

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=5000,output_dim=128))
model.build(input_shape=(None,100))
model.add(LSTM(64))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.compile(loss='binary_crossentropy',optimizer='Adam',metrics=['accuracy'])

#GRU

In [ ]:
from tensorflow.keras.layers import GRU

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=5000,output_dim=128))
model.build(input_shape=(None,100))
model.add(GRU(64))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.compile(loss='binary_crossentropy',optimizer='Adam',metrics=['accuracy'])

In [ ]:
model.fit(x_train,y_train,epochs=10,batch_size=64,verbose=1)

Epoch 1/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 61s 130ms/step - accuracy: 0.6796 - loss: 0.5754
Epoch 2/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 58s 130ms/step - accuracy: 0.8460 - loss: 0.3620
Epoch 3/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 59s 131ms/step - accuracy: 0.8868 - loss: 0.2860
Epoch 4/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 58s 130ms/step - accuracy: 0.9098 - loss: 0.2336
Epoch 5/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 83s 133ms/step - accuracy: 0.9296 - loss: 0.1888
Epoch 6/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 58s 129ms/step - accuracy: 0.9483 - loss: 0.1481
Epoch 7/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 58s 129ms/step - accuracy: 0.9609 - loss: 0.1153
Epoch 8/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 59s 132ms/step - accuracy: 0.9722 - loss: 0.0842
Epoch 9/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 58s 130ms/step - accuracy: 0.9810 - loss: 0.0620
Epoch 10/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 58s 131ms/step - accuracy: 0.9837 - loss: 0.0516


In [ ]:
loss,accuracy = model.evaluate(x_test,y_test)
print("Loss: ",loss)
print("Accuracy: ",accuracy)